In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
df = pd.read_csv(r"C:\Users\niluc\Downloads\PROJECT\Agriculture\Data\kenya_crop_yield.csv")
df.info
df.isnull().sum()


#rain categories
df['rainfall_category'] = pd.cut(df['rainfall_mm'],
                                bins=[0,400,700,1000,9999],
                                labels=['Arid','Semi-Arid','Sub-Humid','Humid'])
#Temparature stress
df['temperature_stress'] = ((df['avg_temp_c'] > 30) | (df['avg_temp_c'] < 12)).astype(int)

#input intensity
df['input_intensity'] = (df['fertiliser_use_kg_ha'] / df['area_harvested_ha']).round(4)

#for trend
df['decade'] = (df['year'] // 10 * 10).astype(str)

df['log_area'] = np.log1p(df['area_harvested_ha'])
df['season_enc'] = df['season'].map({'Long Rains':1, 'Short Rains':0})
df = pd.get_dummies(df, columns=['crop', 'seed_variety','rainfall_category'],
                    drop_first=True)
county_avg = df.groupby('county')['yield_tonnes_ha'].mean()
df['county_avg_yield'] = df['county'].map(county_avg).round(3)

#EDA
fig, axes = plt.subplots(2,3, figsize=(15,9))
#yield distribution
df['yield_tonnes_ha'].hist(bins=40, ax=axes[0,0], color='#1B8CA6', edgecolor='white')
axes[0,0].set_title('Yield distribution (t/ha)')
axes[0,0].axvline(df['yield_tonnes_ha'].mean(), color='red', linestyle ='--',
                 label=f'Mean: {df["yield_tonnes_ha"].mean():.2f}')
axes[0,0].legend()
 # Yield by crop
df.groupby('county')['yield_tonnes_ha'].mean().nlargest(10).plot(
    kind='barh', ax=axes[0,1], color='#2ECC71', edgecolor='white')
axes[0,1].set_title('Top 10 counties by avg Yield')

#rainfall scatter
sample =df.sample(2000, random_state=42)
axes[0,2].scatter(sample['rainfall_mm'], sample['yield_tonnes_ha'],
                  alpha=0.3, s=10,color='#1B8CA6')
axes[0,2].set_title('Rainfall vs Yields')
axes[0,2].set_xlabel('Rainfall(mm)');axes[0,2].set_ylabel('Yields(t/ha)')

#Yield trend over years
df.groupby('year')['yield_tonnes_ha'].mean().plot(
    ax=axes[1,0], color='#F0A500', linewidth=2, marker='o', ms=3)
axes[1,0].set_title('Average Yield Trend')
axes[1,0].set_xlabel('Year');axes[1,0].set_ylabel('Yield(t/ha)')

#season comparison
df.groupby('season')['yield_tonnes_ha'].mean().plot(
    kind='bar',ax=axes[1,1],color=['#1B8CA6','#C0392B'], edgecolor='white')
axes[1,1].set_title('Avg Yield by Season')
axes[1,1].tick_params(axis='x',rotation=0)

#fertilizer vs yields
axes[1,2].scatter(sample['fertiliser_use_kg_ha'],sample['yield_tonnes_ha'],
                  alpha=0.3, s=10,color='#9B59B6')
axes[1,2].set_title('Fertiliser use vs Yields')
axes[1,2].set_xlabel('Fertiliser(kg/ha)')
plt.suptitle('Kenya Agricultural yield EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('agri_EDA', dpi=120)

drop_cols =['county','season','decade','area_harvested_ha']
df = df.drop(drop_cols, axis=1)
bool_cols = df.select_dtypes(include ='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
df.to_csv('agri_clean.csv', index=False)
print('EDA saved.Clean data saved.')




EDA saved.Clean data saved.
